# 05 · Polymorphism, duck typing and protocols

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Day 1 | about 50 minutes live | [04 · Encapsulation and abstraction](04_encapsulation_and_abstraction.ipynb) | [06 · Code safari](06_code_safari_reading_real_code.ipynb) |


Good code that *uses* objects should not care which concrete class it receives. A function that reports a model's errors should work for a simple rule, a scikit-learn model and whatever someone tries next week, without an `if` branch per model type.

This notebook covers three related ideas: one function working with many kinds of objects (**polymorphism**), Python only caring about the methods an object has (**duck typing**), and type hints that describe "anything with a `predict` method" (**protocols**). In the hands-on parts you build a leaderboard that compares every churn model the retention team considers.

> **Learning goals.** By the end you can:
>
> - explain polymorphism with an example from your own model code
> - write one evaluation function that works for your own classes and for scikit-learn models
> - describe the shape of an object with a `typing.Protocol` and say what `@runtime_checkable` does and does not check
> - choose between an abstract base class and a protocol for a given situation and justify it

**Before you start:** notebooks 01 to 04.

**In this notebook**

- Theory 1 · Polymorphism: one interface, many implementations
- Hands-on A · One leaderboard for every churn model
- Theory 2 · Duck typing: Python only checks what you call
- Theory 3 · Protocols: type hints for duck-typed code
- Theory 4 · Mixins: small classes that add one capability
- Hands-on B · What exactly does the leaderboard need?

---

## Setup

Run these cells first. They load and clean the churn data, split it once, and re-create compact versions of classes from earlier notebooks: `ConfusionMatrix` (notebook 01), the rule baselines (notebook 02) and `RetentionCost` (notebook 03). Nothing here is new; the hands-on parts use it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/raw/telco_churn.csv")

customers = pd.read_csv(DATA_PATH).set_index("customerID")  # the customer ID becomes the row label
# TotalCharges is text and blank for customers in their first month (tenure 0): they owe nothing yet.
# to_numeric(..., errors="coerce") turns text that is not a number into NaN; fillna(0.0) replaces NaN by 0
customers["TotalCharges"] = pd.to_numeric(customers["TotalCharges"], errors="coerce").fillna(0.0)
customers["Churn"] = (customers["Churn"] == "Yes").astype(int)  # True/False -> 1/0

X = customers.drop(columns=["Churn"])
y = customers["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"train: {len(X_train)} customers, test: {len(X_test)} customers, churn rate {y.mean():.1%}")

In [ ]:
class ConfusionMatrix:
    """Counts of hits and misses of a binary classifier (1 = churn)."""

    def __init__(self, tn: int, fp: int, fn: int, tp: int):
        self.tn, self.fp, self.fn, self.tp = tn, fp, fn, tp

    @classmethod
    def from_predictions(cls, y_true, y_pred) -> "ConfusionMatrix":
        t, p = np.asarray(y_true), np.asarray(y_pred)  # plain NumPy arrays of 0/1
        # (t == 1) & (p == 0) is True where both hold; .sum() counts the Trues
        return cls(
            tn=int(((t == 0) & (p == 0)).sum()),
            fp=int(((t == 0) & (p == 1)).sum()),
            fn=int(((t == 1) & (p == 0)).sum()),
            tp=int(((t == 1) & (p == 1)).sum()),
        )

    @property
    def precision(self) -> float:
        return self.tp / (self.tp + self.fp) if self.tp + self.fp else 0.0

    @property
    def recall(self) -> float:
        return self.tp / (self.tp + self.fn) if self.tp + self.fn else 0.0

    @property
    def f1(self) -> float:
        p, r = self.precision, self.recall
        return 2 * p * r / (p + r) if p + r else 0.0

    def __repr__(self) -> str:
        return f"ConfusionMatrix(tn={self.tn}, fp={self.fp}, fn={self.fn}, tp={self.tp})"


class RetentionCost:
    """Money lost to churn when the retention team acts on the model's flags.

    Every flagged customer gets an offer (offer_cost). A churner who gets the offer stays with
    probability acceptance_rate. Every churner who leaves costs customer_value.
    """

    def __init__(
        self, offer_cost: float = 50, customer_value: float = 600, acceptance_rate: float = 0.3
    ):
        self.offer_cost = offer_cost
        self.customer_value = customer_value
        self.acceptance_rate = acceptance_rate

    def __call__(self, y_true, y_pred) -> float:
        cm = ConfusionMatrix.from_predictions(y_true, y_pred)
        offers = (cm.tp + cm.fp) * self.offer_cost
        lost_despite_offer = cm.tp * (1 - self.acceptance_rate) * self.customer_value
        lost_unflagged = cm.fn * self.customer_value
        return offers + lost_despite_offer + lost_unflagged


class BaselineModel:
    """Base class of our hand-written rule models (notebook 02)."""

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "BaselineModel":
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        raise NotImplementedError

    def evaluate(self, X: pd.DataFrame, y: pd.Series) -> ConfusionMatrix:
        return ConfusionMatrix.from_predictions(y, self.predict(X))


class MajorityClassModel(BaselineModel):
    """Always predicts the most frequent class seen in training (here: nobody churns)."""

    def fit(self, X, y):
        self.majority_ = int(y.mode()[0])  # .mode()[0] is the most frequent value
        return self

    def predict(self, X):
        return np.full(len(X), self.majority_)  # an array of len(X) copies of the same value


class ContractRuleModel(BaselineModel):
    """The retention team's rule of thumb: flag every month-to-month customer."""

    def predict(self, X):
        # True/False per customer -> 1/0 -> a plain NumPy array
        return (X["Contract"] == "Month-to-month").astype(int).to_numpy()


class TenureRuleModel(BaselineModel):
    """Flag customers who joined less than max_tenure months ago."""

    def __init__(self, max_tenure: int = 12):
        self.max_tenure = max_tenure

    def predict(self, X):
        return (X["tenure"] < self.max_tenure).astype(int).to_numpy()


cost = RetentionCost()
print("Setup done.")

---

## Theory 1 · Polymorphism: one interface, many implementations

*Polymorphism* (Greek for "many shapes") means: code written against an **interface** (a set of method names and what they do) works with every object that provides that interface. You have used it all along: `len()` works on lists, strings, DataFrames and your own classes, because each of them provides `__len__` (notebook 03).

A small, self-contained example: three ways of forecasting tomorrow's number of new sign-ups. They are unrelated classes; all they share is a `predict(history)` method.

In [ ]:
class LastValueForecast:
    """Tomorrow will be like today."""

    def predict(self, history: list[float]) -> float:
        return history[-1]  # the last element of the list


class MeanForecast:
    """Tomorrow will be like the average day so far."""

    def predict(self, history: list[float]) -> float:
        return sum(history) / len(history)


class MovingAverageForecast:
    """Tomorrow will be like the average of the last few days."""

    def __init__(self, window: int = 3):
        self.window = window

    def predict(self, history: list[float]) -> float:
        recent = history[-self.window :]  # the last `window` elements
        return sum(recent) / len(recent)


def forecast_error(model, history: list[float], actual: float) -> float:
    """Absolute error of a one-day-ahead forecast. Works for any object with predict(history)."""
    return abs(model.predict(history) - actual)


daily_signups = [120.0, 135.0, 128.0, 150.0, 162.0]  # new customers per day, last five days
actual_tomorrow = 158.0

for model in [LastValueForecast(), MeanForecast(), MovingAverageForecast(window=3)]:
    print(
        f"{type(model).__name__:22s} error = {forecast_error(model, daily_signups, actual_tomorrow):.1f}"
    )

`forecast_error` never asks *what* it got. It calls `predict`, and each object answers in its own way. Adding a fourth forecasting method next week requires **zero changes** to `forecast_error`. That is the practical value of polymorphism: the code that *uses* objects is decoupled from the code that *implements* them.

---

## Hands-on A · One leaderboard for every churn model

The retention team wants one table that compares every candidate churn model: their current rule of thumb, a logistic regression, a random forest, and whatever someone tries next. Our interface is "has `fit(X, y)` and `predict(X)`". This function relies on nothing else:

In [ ]:
def evaluate(model, X_train, y_train, X_test, y_test) -> ConfusionMatrix:
    """Fit a model on training data and return its confusion matrix on test data."""
    model.fit(X_train, y_train)
    return ConfusionMatrix.from_predictions(y_test, model.predict(X_test))


for model in [MajorityClassModel(), ContractRuleModel(), TenureRuleModel(max_tenure=12)]:
    cm = evaluate(model, X_train, y_train, X_test, y_test)
    print(f"{type(model).__name__:20s} recall={cm.recall:.2f} precision={cm.precision:.2f}")

All three rule models share a base class. scikit-learn models do **not** inherit from our `BaselineModel`, yet the same function works with them.

scikit-learn models need numbers, so each estimator is wrapped in a small scikit-learn `Pipeline` that one-hot encodes the text columns first. You will meet `Pipeline` again in notebook 06; for now treat `make_sklearn_model` as "give me a model that accepts our raw customer table".

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
CATEGORICAL = [
    "Contract",
    "InternetService",
    "PaymentMethod",
    "PaperlessBilling",
    "OnlineSecurity",
    "TechSupport",
]


def make_sklearn_model(estimator):
    """Return a scikit-learn pipeline: encode and scale the columns, then fit `estimator`."""
    preprocess = make_column_transformer(
        (StandardScaler(), NUMERIC),
        (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    )
    return make_pipeline(preprocess, estimator)


models = {
    "contract rule": ContractRuleModel(),
    "logistic regression": make_sklearn_model(LogisticRegression(max_iter=1000)),
    "random forest": make_sklearn_model(
        RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=42)
    ),
}

for name, model in models.items():
    cm = evaluate(model, X_train, y_train, X_test, y_test)
    print(f"{name:20s} recall={cm.recall:.2f} precision={cm.precision:.2f} f1={cm.f1:.2f}")

`evaluate` has no idea that `ContractRuleModel` is a hand-written rule and the others are scikit-learn pipelines.

The numbers also say something: the contract rule catches most churners but flags many loyal customers too; the learned models are more precise. Which one is "best" depends on what an offer costs, which is what `RetentionCost` measures.

### Exercise A.1: the leaderboard

*About 15 minutes, including the discussion.*

Write `leaderboard(models, X_train, y_train, X_test, y_test, cost)`. Then add a fourth model, `make_sklearn_model(GradientBoostingClassifier(random_state=42))`, to the dictionary **without changing your function**.

**What your code should do.**

- `models` is a dict `{name: model}`, e.g. `{"contract rule": ContractRuleModel(), ...}`. Every model has `fit(X, y)` and `predict(X)`.
- For each model: fit it on `X_train, y_train`, predict on `X_test`, build `ConfusionMatrix.from_predictions(y_test, y_pred)` and compute `cost(y_test, y_pred)` (money lost, a number).
- Return a DataFrame with one row per model: the index is the model name, the columns are exactly `precision`, `recall`, `f1`, `cost`, and the rows are sorted by `cost`, lowest first.

**pandas you need.**

- `pd.DataFrame.from_dict(rows, orient="index")`: turns a dict of dicts such as `{"model a": {"precision": 0.5, "recall": 0.8}, "model b": {...}}` into a table with one row per outer key.
- `table.sort_values("cost")`: the same table with rows sorted by the `cost` column, smallest first.

**Discuss.** The scikit-learn models need numbers, the rule models need the raw table, yet `leaderboard` treats them all the same. Where does the knowledge "this model needs encoded columns" live, and what would go wrong if you moved it into `leaderboard` instead?

In [ ]:
# Exercise A.1: your code here

from sklearn.ensemble import GradientBoostingClassifier


def leaderboard(models: dict, X_train, y_train, X_test, y_test, cost) -> pd.DataFrame:
    raise NotImplementedError("Your code here")


all_models = dict(models)  # add the gradient boosting model here

board = leaderboard(all_models, X_train, y_train, X_test, y_test, cost)
board

In [ ]:
# Check for exercise A.1: run after solving it
assert list(board.columns) == ["precision", "recall", "f1", "cost"], board.columns
assert len(board) == 4, "Did you add the gradient boosting model?"
assert board["cost"].is_monotonic_increasing, "Sort by cost, lowest first"
assert board.loc["contract rule", "recall"] > 0.8
print("All checks passed")

<details><summary><b>Show hint</b></summary>

Loop over `models.items()`. For each model call `fit`, then `predict`, build a `ConfusionMatrix`, and
collect a dict like `{"precision": ..., "recall": ..., "f1": ..., "cost": ...}` in a list. Finally
`pd.DataFrame(rows, index=names).sort_values("cost")`.
</details>

<details><summary><b>Show solution</b></summary>

```python
def leaderboard(models: dict, X_train, y_train, X_test, y_test, cost) -> pd.DataFrame:
    rows = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        cm = ConfusionMatrix.from_predictions(y_test, y_pred)
        rows[name] = {
            "precision": cm.precision,
            "recall": cm.recall,
            "f1": cm.f1,
            "cost": cost(y_test, y_pred),
        }
    return pd.DataFrame.from_dict(rows, orient="index").sort_values("cost")


all_models = dict(models)
all_models["gradient boosting"] = make_sklearn_model(GradientBoostingClassifier(random_state=42))
```

The function never mentions a model type. Gradient boosting works because it has `fit` and `predict`.
</details>

---

## Theory 2 · Duck typing: Python only checks what you call

> "If it walks like a duck and quacks like a duck, it is a duck."

Python does not check the *type* of an object before calling a method on it. It looks the method up when the line runs. If the method is there, the call works; if not, you get an `AttributeError` at that moment. This is called **duck typing**.

Two consequences:

- The required "interface" is defined by the code that **uses** the object: whatever methods it calls.
- Mistakes surface late, at the line that makes the call, not when the object is created.

In [ ]:
class SignupReport:
    """A weekly report on sign-ups. It has data, but no predict method."""

    def __init__(self, history: list[float]):
        self.history = history

    def summary(self) -> str:
        return f"{len(self.history)} days, {sum(self.history):.0f} sign-ups"


report = SignupReport(daily_signups)
print(report.summary())

try:
    forecast_error(report, daily_signups, actual_tomorrow)
except AttributeError as exc:
    print(f"AttributeError: {exc}")

> **Check your understanding**
>
> ```python
> class SalesTargetPlan:
>     """The sales team's plan for tomorrow: a fixed number, nothing learned from data."""
>
>     def __init__(self, target: float):
>         self.target = target
>
>     def predict(self, history):
>         return self.target
>
> print(forecast_error(SalesTargetPlan(150.0), daily_signups, actual_tomorrow))
> ```
>
> `SalesTargetPlan` has nothing to do with the forecast classes and ignores the history completely. Does the last line work, and if so, what does it print?
>
> <details><summary><b>Show answer</b></summary>
>
> It works and prints `8.0` (the plan says 150, the actual value is 158). `forecast_error` only calls `predict(history)`, and the plan has such a method. Duck typing does not care whether the object is a "real" forecast, only whether the call works. This is flexible, and it is also why the interface is easy to get wrong without documentation or type hints.
> </details>

In [ ]:
# Predict the output first, then run this cell
class SalesTargetPlan:
    """The sales team's plan for tomorrow: a fixed number, nothing learned from data."""

    def __init__(self, target: float):
        self.target = target

    def predict(self, history):
        return self.target


print(forecast_error(SalesTargetPlan(150.0), daily_signups, actual_tomorrow))

### Libraries are built on conventions

scikit-learn works largely by duck typing plus conventions: `fit` returns `self`, learned attributes end in an underscore (`coef_`, `classes_`), and `predict` accepts the same columns as `fit`. Many of its functions only rely on those conventions. Some need more methods than you might expect; you will see an example in Hands-on B.

---

## Theory 3 · Protocols: type hints for duck-typed code

How should we annotate the `model` parameter of `forecast_error`? A first idea is a common base class:

```python
class Forecast:
    def predict(self, history: list[float]) -> float: ...


def forecast_error(model: Forecast, history: list[float], actual: float) -> float: ...
```

At runtime nothing changes (type hints are not enforced), but a type checker such as `mypy` would reject every forecasting class that does not inherit from `Forecast`: a model from a forecasting library, a colleague's class, our `SalesTargetPlan`.

The hint describes the wrong thing. We do not need "a subclass of `Forecast`"; we need "anything with a `predict(history)` method". The two kinds of compatibility have names:

| Kind | Meaning | Example |
|---|---|---|
| nominal (by inheritance) | the class declares its parent | `ContractRuleModel(BaselineModel)` |
| structural (by shape) | the object has the right methods | `SalesTargetPlan`, a scikit-learn model |

A `typing.Protocol` describes a structural type. Classes do **not** inherit from it; they match it if they have the listed methods.

In [ ]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class Forecaster(Protocol):
    """Anything that forecasts the next value from a history of values."""

    def predict(self, history: list[float]) -> float: ...


def forecast_error(model: Forecaster, history: list[float], actual: float) -> float:
    return abs(model.predict(history) - actual)


for obj in [LastValueForecast(), MovingAverageForecast(), SalesTargetPlan(150.0), report]:
    print(f"{type(obj).__name__:22s} is a Forecaster: {isinstance(obj, Forecaster)}")

None of the forecasting classes mention `Forecaster`, yet they match it. The report does not, because it has no `predict`.

`@runtime_checkable` is what makes `isinstance` work with a protocol. It is a shallow check: it only looks at whether **methods with those names exist**. It does not check arguments or return types:

In [ ]:
class Misleading:
    def predict(self):  # wrong signature: takes no history at all
        return "sometime next week"


print("isinstance(Misleading(), Forecaster):", isinstance(Misleading(), Forecaster))

The full signature check is the job of a static type checker. Running `mypy` on a file that passes `Misleading()` to `forecast_error` reports the mismatch before the code ever runs. That is the main value of protocols: precise type hints for duck-typed code, checked by your editor and by mypy.

### When to use an ABC, when a Protocol

| Situation | Use |
|---|---|
| You own the class family and want to **share code** (a concrete `evaluate` or `fit_transform` in the base class) and force subclasses to implement certain methods | abstract base class (notebook 04) |
| You want to **describe what you need** from objects you do not control (scikit-learn, colleagues' code, test doubles) | `Protocol` |
| You need to check at runtime that an object can be used | prefer calling it and handling the error; `isinstance` with a runtime-checkable protocol only checks method names |

Both can coexist in one project: a base class shares code inside a family you own, and the functions that *use* those objects accept a protocol. The reference package does exactly this: see `ProbabilisticClassifier` in `src/churn/model.py` and `CustomerLoader` in `src/churn/data.py`.

---

## Theory 4 · Mixins: small classes that add one capability

You will see one more pattern constantly in library code: **mixins**. A mixin is a small class that is not meant to be used on its own. It adds one capability to any class that lists it as an extra parent (multiple inheritance, notebook 02).

scikit-learn uses this everywhere. Every classifier inherits `ClassifierMixin`, which provides a `score` method (accuracy) implemented once in terms of `self.predict`:

In [ ]:
from sklearn.base import ClassifierMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# __mro__ lists a class and all its parents in lookup order (notebook 02); [1:5] skips the class itself
print("LogisticRegression parents:", [cls.__name__ for cls in LogisticRegression.__mro__[1:5]])
print("score comes from ClassifierMixin:", LogisticRegression.score is ClassifierMixin.score)
print("same for RandomForestClassifier:", RandomForestClassifier.score is ClassifierMixin.score)

`ClassifierMixin.score` works for every classifier because of polymorphism: it calls `self.predict`, and each classifier brings its own. A class can still override a mixin method: `DummyClassifier`, for example, defines its own `score`. In notebook 06 you will read this kind of code yourself.

Use mixins sparingly in your own code: one capability per mixin, no `__init__` of its own, and name it `...Mixin` so readers know it is not a standalone class.

---

## Hands-on B · What exactly does the leaderboard need?

Back to the churn leaderboard. First, a reminder that duck typing only covers the methods that are actually called. Our rule models work with `evaluate`, but scikit-learn's `cross_val_score` makes a fresh, unfitted copy of the model for every fold (it *clones* it), and cloning needs a method called `get_params`, which our rule models do not have:

In [ ]:
from sklearn.model_selection import cross_val_score

try:
    cross_val_score(TenureRuleModel(max_tenure=12), X_train, y_train, cv=5, scoring="recall")
except TypeError as exc:
    print(f"TypeError: {exc}")

The message is honest: our object "does not seem to be a scikit-learn estimator". Duck typing still applies; the required duck is just bigger than `fit` + `predict`. Inheriting from `sklearn.base.BaseEstimator` adds `get_params` for free; notebook X2 shows how to build fully compatible estimators.

For our own `evaluate`, the protocol from Theory 3 applied to churn models looks like this:

In [ ]:
@runtime_checkable
class Predictor(Protocol):
    """Anything that can be trained and then label customers."""

    def fit(self, X: pd.DataFrame, y: pd.Series) -> object: ...

    def predict(self, X: pd.DataFrame) -> np.ndarray: ...


def evaluate(model: Predictor, X_train, y_train, X_test, y_test) -> ConfusionMatrix:
    model.fit(X_train, y_train)
    return ConfusionMatrix.from_predictions(y_test, model.predict(X_test))


for name, model in models.items():
    print(f"{name:20s} is a Predictor: {isinstance(model, Predictor)}")

### Exercise B.1: a model that inherits from nothing

*About 10 minutes, including the discussion.*

Write a class `AlwaysFlagModel` that flags every customer: recall 1, and the most expensive campaign possible. It must **not** inherit from anything (no parent class in its definition), yet count as a `Predictor`.

**What your code should do.**

- `fit(X, y)` learns nothing and returns the model itself.
- `predict(X)` returns a NumPy array with one `1` per row of `X` (integers), e.g. `array([1, 1, 1])` for three customers.
- `isinstance(AlwaysFlagModel(), Predictor)` is `True`.

**NumPy you need.**

- `np.ones(len(X), dtype=int)`: an array of `len(X)` ones, as integers.

Then add it to `all_models` from exercise A.1 and look at its cost in the leaderboard, next to the do-nothing `MajorityClassModel`.

**Discuss.** `AlwaysFlagModel` counts as a `Predictor` without inheriting from anything. What does `isinstance(obj, Predictor)` really prove, and what does it not? Is the cheapest model in your leaderboard one of the two extremes?

In [ ]:
# Exercise B.1: your code here


class AlwaysFlagModel: ...

In [ ]:
# Check for exercise B.1: run after solving it
assert isinstance(AlwaysFlagModel(), Predictor)
cm = evaluate(AlwaysFlagModel(), X_train, y_train, X_test, y_test)
assert cm.recall == 1.0 and cm.tn == 0
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
class AlwaysFlagModel:
    """Send an offer to everyone: maximum recall, maximum campaign cost."""

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "AlwaysFlagModel":
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return np.ones(len(X), dtype=int)
```

`AlwaysFlagModel` matches `Predictor` purely by its shape. In the leaderboard neither extreme (flag everybody, flag nobody) is the cheapest.
</details>

### Exercise B.2: a protocol for the cost

> If you finish early, try this one.

`leaderboard` needs two kinds of objects: models and a cost. Describe both with protocols.

**What your code should do.**

- Define a runtime-checkable protocol `CostFunction` with a single method `__call__(self, y_true: pd.Series, y_pred: np.ndarray) -> float` (notebook 03: `__call__` is what makes `cost(...)` work).
- `isinstance(cost, CostFunction)` is `True` for our `RetentionCost` object, without changing `RetentionCost`. A `ConfusionMatrix` is not a `CostFunction`.
- Copy your `leaderboard` from exercise A.1 into the cell and annotate it: `models: dict[str, Predictor]`, `cost: CostFunction`, the data parameters as `pd.DataFrame` / `pd.Series`, return type `pd.DataFrame`. The body does not change.

If you have time, save your function and models in a file, e.g. `playground/compare.py`, run `uv run mypy playground/compare.py` in the terminal, then pass a `Misleading()` object to your function and see what mypy says.

**Discuss.** For the models your team writes, would you rather give them a common base class, only a protocol, or both? What does each choice make easy, and what does it make hard?

In [ ]:
# Exercise B.2: your code here


class CostFunction: ...

In [ ]:
# Check for exercise B.2: run after solving it
assert isinstance(cost, CostFunction), "RetentionCost should match CostFunction by its shape"
assert not isinstance(ConfusionMatrix(1, 1, 1, 1), CostFunction)
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
@runtime_checkable
class CostFunction(Protocol):
    def __call__(self, y_true: pd.Series, y_pred: np.ndarray) -> float: ...


def leaderboard(
    models: dict[str, Predictor],
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    cost: CostFunction,
) -> pd.DataFrame:
    rows = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        cm = ConfusionMatrix.from_predictions(y_test, y_pred)
        rows[name] = {"precision": cm.precision, "recall": cm.recall, "f1": cm.f1, "cost": cost(y_test, y_pred)}
    return pd.DataFrame.from_dict(rows, orient="index").sort_values("cost")
```

A protocol with only `__call__` describes "something you can call like this function". Notebook 07 shows the shorter spelling for exactly this case, `Callable[[pd.Series, np.ndarray], float]`.
</details>

### Exercise B.3: choose each model's threshold

> **Challenge**

The leaderboard compares models at their default decision rule. But logistic regression and random forests produce probabilities (`predict_proba`), and the retention team can pick any threshold. A model that loses at threshold 0.5 may win at 0.25. Write `best_threshold(model, X_val, y_val, cost)`, then build a leaderboard that uses each model's best threshold.

**What your code should do.**

- If `model` has a `predict_proba` method: try the thresholds 0.05, 0.10, ..., 0.95 on the validation data, compute the cost of each, and return the threshold (a float) with the lowest cost.
- If it has no `predict_proba` (our rule models): return `None`.
- Important: do not choose the threshold on the test set; you would be tuning on the data you report. The check cell splits `X_train` once more into a fitting part and a validation part.

**pandas and NumPy you need.**

- `hasattr(model, "predict_proba")`: `True` if the object has an attribute or method with that name.
- `model.predict_proba(X_val)[:, 1]`: one row per customer and one column per class; `[:, 1]` takes column 1, the probability of churn.
- `np.arange(0.05, 0.96, 0.05)`: the array `[0.05, 0.10, ..., 0.95]`.
- `(proba >= t).astype(int)`: 0/1 decisions at threshold `t`.
- `np.argmin(costs)`: the position of the smallest value in a list; use it to pick the matching threshold.

**Discuss.** Talk these through with your partner or the group; if you work alone, answer them in a markdown cell:
1. Checking `hasattr(model, "predict_proba")` is duck typing at runtime. What would a second protocol `ProbabilisticPredictor` add, and what would it not add?
2. Build `make_sklearn_model(LinearSVC())` (from `sklearn.svm`; a classifier without probabilities). `Pipeline` defines a `predict_proba` method in its class, so what do you expect `hasattr(pipeline, "predict_proba")` to return? Try it. What does the result teach you about writing wrapper classes (like `ChurnModel`) that other code will duck-type?

In [ ]:
# Exercise B.3: your code here


def best_threshold(model, X_val, y_val, cost) -> float | None:
    raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise B.3: run after solving it
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)
lr = make_sklearn_model(LogisticRegression(max_iter=1000)).fit(X_fit, y_fit)
t = best_threshold(lr, X_val, y_val, cost)
assert t is not None and 0.05 <= t <= 0.95
assert best_threshold(ContractRuleModel().fit(X_fit, y_fit), X_val, y_val, cost) is None
tuned_cost = cost(y_test, (lr.predict_proba(X_test)[:, 1] >= t).astype(int))
print(
    f"best threshold {t:.2f}: test cost {tuned_cost:,.0f} vs {cost(y_test, lr.predict(X_test)):,.0f} at 0.5"
)
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
def best_threshold(model, X_val, y_val, cost) -> float | None:
    if not hasattr(model, "predict_proba"):
        return None
    proba = model.predict_proba(X_val)[:, 1]
    thresholds = np.arange(0.05, 0.96, 0.05)
    costs = [cost(y_val, (proba >= t).astype(int)) for t in thresholds]
    return float(thresholds[int(np.argmin(costs))])
```

1. A protocol `ProbabilisticPredictor` (with `predict_proba`) documents the requirement and lets mypy
   check calls before running anything. It does not add a runtime guarantee: `isinstance` with
   `@runtime_checkable` still only checks that a method with that name exists.
2. It returns `False`, although the method is written in the `Pipeline` class. scikit-learn wraps it
   with a decorator (`available_if`) that makes the attribute disappear when the final step lacks
   `predict_proba`. The library goes out of its way to keep `hasattr` truthful, because so much code
   duck-types on it. A naive wrapper that always defines `predict_proba` and forwards the call would
   make `hasattr` say `True` and then fail when called. Lessons: a wrapper should only advertise the
   methods it can really serve, and when in doubt the robust duck-typing style is "try it and handle
   the error" (EAFP: easier to ask forgiveness than permission).

With the cost model used here, the cheapest threshold for logistic regression is below 0.5:
missing a churner (600) is far more expensive than an unnecessary offer (50).
</details>

---

## Summary

- **Polymorphism**: code that calls methods of an interface works with every object that provides them.
  One `forecast_error`, `evaluate` or `leaderboard` function serves many unrelated classes.
- **Duck typing**: Python checks for a method only when the call happens. The "interface" is whatever the
  calling code uses. Errors appear late, at the call.
- Some library functions need more than `fit`/`predict` (e.g. cloning needs `get_params`). Read the error,
  it usually says what is missing.
- **Protocols** give duck-typed code precise type hints. Classes match them by shape, without inheriting.
  `@runtime_checkable` only checks method names.
- Use an **ABC** to share code inside a family you own; use a **Protocol** to describe what you need from
  objects you do not control.
- **Mixins** add one capability to many classes (`ClassifierMixin.score`).

**Common pitfalls**
- Annotating with your own base class when you mean "anything with these methods".
- Trusting `isinstance(obj, SomeProtocol)` as proof that calls will succeed.
- Writing `if type(model) == ...` branches instead of giving each model the method you need.

## Going further

- Notebook X2 builds estimators that work with `cross_val_score` and `GridSearchCV`.

---

## Sources and further reading

**Sources.** This notebook is based on:

- Python docs: [`typing.Protocol`](https://docs.python.org/3/library/typing.html#typing.Protocol) and the glossary entry [duck typing](https://docs.python.org/3/glossary.html#term-duck-typing).
- [PEP 544, Protocols: Structural subtyping](https://peps.python.org/pep-0544/), the design document with its motivation.
- [mypy: Protocols and structural subtyping](https://mypy.readthedocs.io/en/stable/protocols.html).
- [scikit-learn: Developing scikit-learn estimators](https://scikit-learn.org/stable/developers/develop.html), the conventions our duck typing relies on.

**For self-study.** Where to go next if you want to learn more:

- [Real Python: Duck Typing in Python](https://realpython.com/duck-typing-python/) (article, intermediate).
- [Real Python: Python Protocols, Leveraging Structural Subtyping](https://realpython.com/python-protocol/) (article, intermediate).
- Luciano Ramalho, *Fluent Python*, 2nd edition, O'Reilly 2022 ([book site](https://www.fluentpython.com/)), chapter 13 "Interfaces, Protocols, and ABCs" (book, advanced): the four ways of typing in Python and when to use each.

---

[← 04 · Encapsulation and abstraction](04_encapsulation_and_abstraction.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [06 · Code safari →](06_code_safari_reading_real_code.ipynb)